In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
dbutils.widgets.text("catalogo", "etl_supermercado")
dbutils.widgets.text("esquema_source", "silver")
dbutils.widgets.text("esquema_sink", "golden")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
df = spark.table(f"{catalogo}.{esquema_source}.ventas_detalle")

In [0]:
df_dept = df.groupBy("department") \
    .agg(
        count("product_id").alias("total_productos_pedidos"),
        countDistinct("order_id").alias("total_ordenes"),
        sum(when(col("es_recompra"), 1).otherwise(0)).alias("total_recompras"),
    ) \
    .withColumn("pct_recompra",round(col("total_recompras") / col("total_productos_pedidos") * 100, 2)) \
    .withColumn("promedio_productos_por_orden",round(col("total_productos_pedidos") / col("total_ordenes"), 2)) \
    .withColumn("ingestion_date", current_timestamp())

df_dept.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.ventas_por_departamento")

In [0]:
df_top = (df.groupBy("product_id", "product_name", "aisle", "department")
    .agg(
        count("order_id").alias("total_pedidos"),
        sum(when(col("es_recompra"), 1).otherwise(0)).alias("total_recompras"),
    )
    .withColumn("pct_recompra",round(col("total_recompras") / col("total_pedidos") * 100, 2))
    .withColumn("rank_popularidad",dense_rank().over(Window.orderBy(col("total_pedidos").desc())))
    .filter(col("rank_popularidad") <= 20)
    .withColumn("ingestion_date", current_timestamp())
)

df_top.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.top_productos")

In [0]:
def tipo_cliente(n):
    if n is None:    return "Desconocido"
    elif n == 1:     return "Nuevo"
    elif n <= 5:     return "Ocasional"
    elif n <= 15:    return "Frecuente"
    else:            return "Fiel"

tipo_udf = F.udf(tipo_cliente, StringType())

df_usuarios = df.groupBy("user_id") \
    .agg(
        countDistinct("order_id")             .alias("total_ordenes"),
        avg("days_since_prior_order")         .alias("promedio_dias_entre_ordenes"),
        countDistinct("product_id")           .alias("total_productos_distintos"),
    ) \
    .withColumn("promedio_dias_entre_ordenes", round(col("promedio_dias_entre_ordenes"), 1)) \
    .withColumn("tipo_cliente", tipo_udf(col("total_ordenes"))) \
    .withColumn("ingestion_date", current_timestamp())

df_usuarios.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.frecuencia_usuarios")